## Notebook objective
This notebook analyzes false positives produced by definition-generation models. Using parquet evaluation reports, it identifies the lemmas and individual instances that triggered spurious detections.

In [ ]:
import pandas as pd

### Input data
The analysis uses one parquet evaluation report per model, containing lemma-level results and the false-positive senses detected for each instance.

In [ ]:
mistral_df = pd.read_parquet(
    "../resources/generated/evaluation-report/definition-generator-v2/Qwen3-Embedding-4B/mistral-large-2512_eval.parquet"
).sort_values(by="false_positives", ascending=False)
claude_df: pd.DataFrame = pd.read_parquet(
    "../resources/generated/evaluation-report/definition-generator-v2/Qwen3-Embedding-4B/claude-sonnet-4.6_eval.parquet"
).sort_values(by="false_positives", ascending=False)
deepseek_df: pd.DataFrame = pd.read_parquet(
    "../resources/generated/evaluation-report/definition-generator-v2/Qwen3-Embedding-4B/deepseek-v3.2_eval.parquet"
).sort_values(by="false_positives", ascending=False)
gemini_df: pd.DataFrame = pd.read_parquet(
    "../resources/generated/evaluation-report/definition-generator-v2/Qwen3-Embedding-4B/gemini-2.5-flash_eval.parquet"
).sort_values(by="false_positives", ascending=False)
llama_df: pd.DataFrame = pd.read_parquet(
    "../resources/generated/evaluation-report/definition-generator-v2/Qwen3-Embedding-4B/llama-4-scout_eval.parquet"
).sort_values(by="false_positives", ascending=False)
qwen3_df: pd.DataFrame = pd.read_parquet(
    "../resources/generated/evaluation-report/definition-generator-v2/Qwen3-Embedding-4B/qwen3.6-plus_eval.parquet"
).sort_values(by="false_positives", ascending=False)

In [32]:
mistral_df.head()

,lemma,similarity,matched_similarity,false_negatives,false_positives,false_positives_instances
293,prendere,0.706796,0.706796,0,66,"[{'definition': 'scrivere o annotare qualcosa,..."
82,portare,0.719141,0.719141,0,40,[{'definition': 'Indossare un capo di abbiglia...
74,nero,0.750000,0.750000,0,35,[{'definition': 'Relativo a contenuti o temi o...
79,testa,0.750355,0.750355,0,32,[{'definition': 'Persona che guida o comanda u...
120,rete,0.782813,0.782813,0,32,[{'definition': 'Sistema di collegamenti tra p...


In [ ]:
def save_df(df: pd.DataFrame, model: str) -> None:
    df = df.iloc[:10]
    df = df.explode("false_positives_instances").reset_index(drop=True)
    df["predicted_sense"] = df["false_positives_instances"].apply(
        lambda x: x["definition"]
    )
    df["predicted_index"] = df["false_positives_instances"].apply(lambda x: x["index"])
    df.drop(columns=["false_positives_instances"], inplace=True)
    df.to_excel(
        f"Lemmas with most FP senses per model/{model}_top_10.xlsx", index=False
    )

### Exporting the most problematic cases
The helper function below selects the top-ranked lemmas, expands the nested false-positive instances into tabular form, and exports them to Excel files for inspection.

In [35]:
save_df(mistral_df, "mistral")
save_df(claude_df, "claude")
save_df(deepseek_df, "deepseek")
save_df(gemini_df, "gemini")
save_df(llama_df, "llama")
save_df(qwen3_df, "qwen3")